# VRA/ANAC — ELT de 2019 (do zero ao resultado tratado)

**Projeto de Integração — Banco de Dados (UFPE).** Notebook reproduzível com os 12 arquivos mensais de 2019.

**Fluxo:** `CSV ZIP → leitura fiel → PostgreSQL (vra_bruta) → SQL (duplicatas / rejeições / conversões / métricas) → vra_tratada`.

**Escopo deste notebook:** somente o **ELT de 2019**. Não cria dimensões nem `fato_voo` ainda, para não conflitar com o modelo conjunto dos anos 2020 e 2021.

**Contrato de qualidade:** remover duplicatas exatas excedentes na camada analítica, rejeitar registros sem campos obrigatórios, preservar registros originais para auditoria, sinalizar variações de horário absolutas acima de 24h. **Nunca apagar da camada bruta.**

> O PostgreSQL é instalado apenas no ambiente temporário do Colab; não é preciso instalar nada no computador. A sessão do Colab pode ser reiniciada e perder o banco: o notebook permite reconstruí-lo.

## 1. Instalar PostgreSQL no ambiente Colab
Execute uma vez por sessão. Estas células foram feitas para ambiente Linux do Google Colab, não para Python local do Windows.

In [1]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-contrib", "-qq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"], check=True)
subprocess.run(["service", "postgresql", "start"], check=True)
print("PostgreSQL instalado e iniciado no Colab.")

PostgreSQL instalado e iniciado no Colab.


## 2. Conectar-se ao banco
Senha temporária gerada em execução, sem mostrar no notebook. Os scripts criam apenas o schema `elt_2019`.

In [2]:
import secrets
import psycopg2
import subprocess

senha_pg = secrets.token_urlsafe(24)
# Envía a senha ao psql por stdin (evita registrá-la na linha de comando)
subprocess.run(
    ["runuser", "-u", "postgres", "--", "psql", "-v", "ON_ERROR_STOP=1"],
    input="ALTER USER postgres PASSWORD '" + senha_pg.replace("'", "''") + "';\n",
    text=True, check=True, capture_output=True
)
conn = psycopg2.connect(host="localhost", port=5432, dbname="postgres", user="postgres", password=senha_pg)
conn.autocommit = False
print("Conexão PostgreSQL funcionando!")

Conexão PostgreSQL funcionando!


## 3. Upload da base original
Faça upload do ZIP com os **12 arquivos**. O upload é usado somente para ler/transportar os registros: nada é transformado neste momento.

In [3]:
from google.colab import files
import zipfile
import io
import pandas as pd

arquivos_enviados = files.upload()
zips = [(nome, dados) for nome, dados in arquivos_enviados.items() if nome.lower().endswith(".zip")]
assert len(zips) == 1, "Envie exatamente um arquivo ZIP de 2019."
nome_zip, bytes_zip = zips[0]
print("Arquivo enviado:", nome_zip)

Saving Base de dados 2019.zip to Base de dados 2019.zip
Arquivo enviado: Base de dados 2019.zip


## 4. Extrair os CSVs e adicionar rastreabilidade
Confere os 20 cabeçalhos originais, mantém **todos os valores como texto** e adiciona as duas colunas técnicas de origem. Não remove nenhuma linha.

In [4]:
# Colunas oficiais identificadas nos 12 arquivos de 2019
colunas_esperadas = [
    "Sigla ICAO Empresa Aérea", "Empresa Aérea", "Número Voo", "Código DI",
    "Código Tipo Linha", "Modelo Equipamento", "Número de Assentos",
    "Sigla ICAO Aeroporto Origem", "Descrição Aeroporto Origem", "Partida Prevista",
    "Partida Real", "Sigla ICAO Aeroporto Destino", "Descrição Aeroporto Destino",
    "Chegada Prevista", "Chegada Real", "Situação Voo", "Justificativa",
    "Referência", "Situação Partida", "Situação Chegada"
]

dataframes = []
with zipfile.ZipFile(io.BytesIO(bytes_zip)) as z:
    nomes = sorted(p for p in z.namelist() if p.lower().endswith(".csv"))
    assert len(nomes) == 12, f"Esperados 12 CSVs, encontrados {len(nomes)}"
    for nome in nomes:
        with z.open(nome) as arquivo:
            df = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig", dtype=str,
                             keep_default_na=False)
        assert list(df.columns) == colunas_esperadas, f"Cabeçalho divergente: {nome}"
        df["arquivo_origem"] = nome
        df["linha_origem"] = range(2, len(df) + 2)
        dataframes.append(df)
        print(f"{nome}: {len(df):,} linhas")

df_2019 = pd.concat(dataframes, ignore_index=True)
print("Total:", len(df_2019), "linhas e", len(df_2019.columns), "colunas")
assert len(df_2019) == 983414, "Total diferente da base de referência; confira o ZIP."
display(df_2019.head(3))

Base de dados 2019/VRA_2019_01.csv: 91,914 linhas
Base de dados 2019/VRA_2019_02.csv: 79,571 linhas
Base de dados 2019/VRA_2019_03.csv: 83,152 linhas
Base de dados 2019/VRA_2019_04.csv: 77,445 linhas
Base de dados 2019/VRA_2019_05.csv: 76,732 linhas
Base de dados 2019/VRA_2019_06.csv: 74,254 linhas
Base de dados 2019/VRA_2019_07.csv: 85,687 linhas
Base de dados 2019/VRA_2019_08.csv: 80,588 linhas
Base de dados 2019/VRA_2019_09.csv: 80,537 linhas
Base de dados 2019/VRA_2019_10.csv: 84,268 linhas
Base de dados 2019/VRA_2019_11.csv: 81,046 linhas
Base de dados 2019/VRA_2019_12.csv: 88,220 linhas
Total: 983414 linhas e 22 colunas


,Sigla ICAO Empresa Aérea,Empresa Aérea,Número Voo,Código DI,Código Tipo Linha,Modelo Equipamento,Número de Assentos,Sigla ICAO Aeroporto Origem,Descrição Aeroporto Origem,Partida Prevista,...,Descrição Aeroporto Destino,Chegada Prevista,Chegada Real,Situação Voo,Justificativa,Referência,Situação Partida,Situação Chegada,arquivo_origem,linha_origem
0,AAF,AIGLE AZUR,0035,0,I,A332,283,LFPO,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,25/01/2019 06:15,...,VIRACOPOS - CAMPINAS - SP - BRASIL,25/01/2019 18:15,25/01/2019 18:15,REALIZADO,,2019-01-25,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,2
1,AAF,AIGLE AZUR,0036,0,I,A332,283,SBKP,VIRACOPOS - CAMPINAS - SP - BRASIL,25/01/2019 20:15,...,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,26/01/2019 07:45,26/01/2019 07:45,REALIZADO,,2019-01-25,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,3
2,AAF,AIGLE AZUR,0037,0,I,A332,283,LFPO,PARIS ORLY AIRPORT - ORLY (NEAR PARIS) - FRANÇA,26/01/2019 09:45,...,VIRACOPOS - CAMPINAS - SP - BRASIL,26/01/2019 21:45,26/01/2019 21:45,REALIZADO,,2019-01-26,Pontual,Pontual,Base de dados 2019/VRA_2019_01.csv,4


## 5. Criar a tabela de **carga bruta** no PostgreSQL
A tabela mantém os **nomes e valores originais das 20 colunas**, além de `arquivo_origem` e `linha_origem`. A identificação SQL vai entre aspas devido aos acentos e espaços. Reiniciar o pipeline apaga e recria **somente** a tabela bruta e suas derivadas neste schema.

In [5]:
from psycopg2 import sql

with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS elt_2019")
    # Ao rerodar, desfaz apenas os objetos gerados por ESTE pipeline.
    for tabela in ["vra_tratada", "vra_rejeicoes", "vra_duplicatas", "vra_classificada", "vra_bruta"]:
        cur.execute(sql.SQL("DROP TABLE IF EXISTS elt_2019.{} CASCADE").format(sql.Identifier(tabela)))
    definicoes = [
        sql.SQL("{} {}").format(sql.Identifier(col), sql.SQL("INTEGER" if col == "linha_origem" else "TEXT"))
        for col in df_2019.columns
    ]
    cur.execute(sql.SQL("CREATE TABLE elt_2019.vra_bruta ({})").format(sql.SQL(", ").join(definicoes)))
conn.commit()
print("Schema e tabela de carga bruta criados.")

Schema e tabela de carga bruta criados.


## 6. Carregar os dados com `COPY` (LOAD)
Esta é a etapa **L** do ELT. Nenhuma transformação de negócio aqui: a carga é feita em lotes e validada depois. `COPY` preserva os textos vazios, incluindo `""`.

In [6]:
import csv

colunas_sql = sql.SQL(", ").join(sql.Identifier(c) for c in df_2019.columns)
comando_copy = sql.SQL(
    "COPY elt_2019.vra_bruta ({}) FROM STDIN WITH (FORMAT CSV, NULL '\\N')"
).format(colunas_sql)
# A string SQL deverá conter NULL '\N' (marcador que não ocorre nos campos originais)
print("Comando COPY preparado.")

lote = 25000
try:
    with conn.cursor() as cur:
        for inicio in range(0, len(df_2019), lote):
            parte = df_2019.iloc[inicio:inicio+lote]
            dados_csv = io.StringIO()
            parte.to_csv(dados_csv, sep=",", index=False, header=False,
                         lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
            dados_csv.seek(0)
            cur.copy_expert(comando_copy.as_string(conn), dados_csv)
    conn.commit()
except Exception:
    conn.rollback()
    raise
print("LOAD concluído.")

Comando COPY preparado.
LOAD concluído.


## 7. Validar a carga bruta
Confere o total e a distribuição por arquivo; se a validação falhar, não siga para a transformação.

In [7]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM elt_2019.vra_bruta")
    total_pg = cur.fetchone()[0]
    cur.execute("SELECT arquivo_origem, COUNT(*) FROM elt_2019.vra_bruta GROUP BY 1 ORDER BY 1")
    contagens = cur.fetchall()
print("Linhas no PostgreSQL:", total_pg)
for nome, n in contagens: print(nome, n)
assert total_pg == len(df_2019) == 983414
assert len(contagens) == 12
print("Carga bruta validada.")

Linhas no PostgreSQL: 983414
Base de dados 2019/VRA_2019_01.csv 91914
Base de dados 2019/VRA_2019_02.csv 79571
Base de dados 2019/VRA_2019_03.csv 83152
Base de dados 2019/VRA_2019_04.csv 77445
Base de dados 2019/VRA_2019_05.csv 76732
Base de dados 2019/VRA_2019_06.csv 74254
Base de dados 2019/VRA_2019_07.csv 85687
Base de dados 2019/VRA_2019_08.csv 80588
Base de dados 2019/VRA_2019_09.csv 80537
Base de dados 2019/VRA_2019_10.csv 84268
Base de dados 2019/VRA_2019_11.csv 81046
Base de dados 2019/VRA_2019_12.csv 88220
Carga bruta validada.


## 8. Transformar **dentro do PostgreSQL** (Transform)

A partir daqui tudo é feito em SQL. O script:

1. Sinaliza cada ocorrência usando `ROW_NUMBER()` nos **20 campos originais**, ignorando metadados.
2. Guarda as 470 ocorrências excedentes em `vra_duplicatas` e as remove apenas da **tabela analítica**.
3. Guarda rejeições em `vra_rejeicoes`, como datas previstas ausentes ou aeroporto obrigatório ausente.
4. Padroniza nomes e textos, converte vazios em `NULL`, preserva número do voo como texto.
5. Converte datas, calcula diferenças e atrasos em minutos, sinaliza diferenças absolutas maiores que 1.440 minutos.

O script completo também será entregue separadamente como `ELT_ANAC_2019_transformacoes.sql`, para o GitHub. A célula seguinte contém o SQL sem depender de arquivo local.

In [8]:
# 20 campos usados como chave de comparação EXATA, antes da padronização
colunas_origem = colunas_esperadas
particoes = sql.SQL(", ").join(sql.Identifier(c) for c in colunas_origem)

# SQL da classificação e auditoria; NÃO altera vra_bruta.
with conn.cursor() as cur:
    cur.execute("SET TIME ZONE 'UTC'")
    cur.execute("DROP TABLE IF EXISTS elt_2019.vra_tratada CASCADE")
    cur.execute("DROP TABLE IF EXISTS elt_2019.vra_rejeicoes CASCADE")
    cur.execute("DROP TABLE IF EXISTS elt_2019.vra_duplicatas CASCADE")
    cur.execute("DROP TABLE IF EXISTS elt_2019.vra_classificada CASCADE")
    cur.execute(sql.SQL("""
        CREATE TABLE elt_2019.vra_classificada AS
        SELECT r.*, ROW_NUMBER() OVER (
            PARTITION BY {}
            ORDER BY arquivo_origem, linha_origem
        ) AS ocorrencia_num
        FROM elt_2019.vra_bruta r
    """).format(particoes))
    cur.execute("""
        CREATE TABLE elt_2019.vra_duplicatas AS
        SELECT * FROM elt_2019.vra_classificada
        WHERE ocorrencia_num > 1
    """)
    cur.execute("""
        CREATE TABLE elt_2019.vra_rejeicoes AS
        SELECT *,
          CASE
            WHEN NULLIF(BTRIM("Partida Prevista"), '') IS NULL
             AND NULLIF(BTRIM("Chegada Prevista"), '') IS NULL
               THEN 'PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES'
            WHEN NULLIF(BTRIM("Partida Prevista"), '') IS NULL THEN 'PARTIDA_PREVISTA_AUSENTE'
            WHEN NULLIF(BTRIM("Chegada Prevista"), '') IS NULL THEN 'CHEGADA_PREVISTA_AUSENTE'
            WHEN NULLIF(BTRIM("Sigla ICAO Empresa Aérea"), '') IS NULL THEN 'COMPANHIA_AUSENTE'
            WHEN NULLIF(BTRIM("Sigla ICAO Aeroporto Origem"), '') IS NULL THEN 'AEROPORTO_ORIGEM_AUSENTE'
            WHEN NULLIF(BTRIM("Sigla ICAO Aeroporto Destino"), '') IS NULL THEN 'AEROPORTO_DESTINO_AUSENTE'
          END AS motivo_rejeicao
        FROM elt_2019.vra_classificada
        WHERE ocorrencia_num = 1 AND (
           NULLIF(BTRIM("Partida Prevista"), '') IS NULL
        OR NULLIF(BTRIM("Chegada Prevista"), '') IS NULL
        OR NULLIF(BTRIM("Sigla ICAO Empresa Aérea"), '') IS NULL
        OR NULLIF(BTRIM("Sigla ICAO Aeroporto Origem"), '') IS NULL
        OR NULLIF(BTRIM("Sigla ICAO Aeroporto Destino"), '') IS NULL
        )
    """)
conn.commit()
print("Classificação SQL, auditoria de duplicatas e rejeições criadas.")

Classificação SQL, auditoria de duplicatas e rejeições criadas.


## 9. Criar a camada tratada SQL
Esta célula converte as colunas de modo uniforme. O PostgreSQL é quem faz a transformação, não o Pandas. **Datas permanecem sem conversão de fuso**. A origem foi previamente verificada: datas preenchidas de 2019 seguem o formato previsto.

In [9]:
# SQL explícito construído a partir do contrato, executado somente no PostgreSQL
mapeamento = [
    ("Sigla ICAO Empresa Aérea", "companhia_icao", "codigo"),
    ("Empresa Aérea", "companhia_nome", "codigo"),
    ("Número Voo", "numero_voo", "texto"),
    ("Código DI", "codigo_di", "codigo"),
    ("Código Tipo Linha", "tipo_linha", "codigo"),
    ("Modelo Equipamento", "modelo_equipamento", "codigo"),
    ("Número de Assentos", "numero_assentos", "inteiro"),
    ("Sigla ICAO Aeroporto Origem", "aeroporto_origem", "codigo"),
    ("Descrição Aeroporto Origem", "descricao_origem", "texto"),
    ("Partida Prevista", "partida_prevista", "datahora"),
    ("Partida Real", "partida_real", "datahora"),
    ("Sigla ICAO Aeroporto Destino", "aeroporto_destino", "codigo"),
    ("Descrição Aeroporto Destino", "descricao_destino", "texto"),
    ("Chegada Prevista", "chegada_prevista", "datahora"),
    ("Chegada Real", "chegada_real", "datahora"),
    ("Situação Voo", "situacao_voo", "codigo"),
    ("Justificativa", "justificativa", "texto"),
    ("Referência", "referencia", "data"),
    ("Situação Partida", "situacao_partida", "codigo"),
    ("Situação Chegada", "situacao_chegada", "codigo"),
]

expressions = []
for original, destino, tipo in mapeamento:
    nome = sql.Identifier(original)
    if tipo == "codigo": expr = sql.SQL("UPPER(NULLIF(BTRIM({}), ''))").format(nome)
    elif tipo == "texto": expr = sql.SQL("NULLIF(BTRIM({}), '')").format(nome)
    elif tipo == "inteiro": expr = sql.SQL("CASE WHEN BTRIM({0}) ~ '^[0-9]+$' THEN BTRIM({0})::integer ELSE NULL END").format(nome)
    elif tipo == "datahora":
        expr = sql.SQL("CASE WHEN BTRIM({0}) ~ '^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}} [0-9]{{2}}:[0-9]{{2}}$' THEN TO_TIMESTAMP(BTRIM({0}), 'DD/MM/YYYY HH24:MI')::timestamp ELSE NULL END").format(nome)
    else:
        expr = sql.SQL("CASE WHEN BTRIM({0}) ~ '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$' THEN TO_DATE(BTRIM({0}), 'YYYY-MM-DD') ELSE NULL END").format(nome)
    expressions.append(sql.SQL("{} AS {}").format(expr, sql.Identifier(destino)))

query = sql.SQL("""
    CREATE TABLE elt_2019.vra_tratada AS
    WITH dados AS (
        SELECT {}, arquivo_origem, linha_origem
        FROM elt_2019.vra_classificada c
        WHERE ocorrencia_num = 1
          AND NOT EXISTS (
              SELECT 1 FROM elt_2019.vra_rejeicoes r
              WHERE r.arquivo_origem = c.arquivo_origem
                AND r.linha_origem = c.linha_origem
          )
    ), medidas AS (
        SELECT *,
            EXTRACT(EPOCH FROM (partida_real - partida_prevista)) / 60 AS diferenca_partida_min,
            EXTRACT(EPOCH FROM (chegada_real - chegada_prevista)) / 60 AS diferenca_chegada_min
        FROM dados
    ), atrasos AS (
        SELECT *,
            GREATEST(diferenca_partida_min, 0) AS atraso_partida_min,
            GREATEST(diferenca_chegada_min, 0) AS atraso_chegada_min
        FROM medidas
    )
    SELECT *,
      (diferenca_partida_min IS NOT NULL AND ABS(diferenca_partida_min) > 1440) AS partida_atipica_flag,
      (diferenca_chegada_min IS NOT NULL AND ABS(diferenca_chegada_min) > 1440) AS chegada_atipica_flag
    FROM atrasos
""").format(sql.SQL(",\n            ").join(expressions))

try:
    with conn.cursor() as cur:
        cur.execute(query)
    conn.commit()
except Exception:
    conn.rollback()
    raise
print("Tabela SQL elt_2019.vra_tratada criada.")

Tabela SQL elt_2019.vra_tratada criada.


## 10. Auditoria final e reconciliação
A regra obrigatória é: **bruta = duplicatas excedentes + rejeições não duplicadas + tratada**. Também medimos os flags de horários acima de 24h. Na base original validada, a meta é **982.926** registros tratados.

In [10]:
import pandas as pd
with conn.cursor() as cur:
    cur.execute("""
        SELECT
          (SELECT COUNT(*) FROM elt_2019.vra_bruta),
          (SELECT COUNT(*) FROM elt_2019.vra_duplicatas),
          (SELECT COUNT(*) FROM elt_2019.vra_rejeicoes),
          (SELECT COUNT(*) FROM elt_2019.vra_tratada),
          (SELECT COUNT(*) FROM elt_2019.vra_tratada WHERE partida_atipica_flag),
          (SELECT COUNT(*) FROM elt_2019.vra_tratada WHERE chegada_atipica_flag)
    """)
    bruta, duplicatas, rejeicoes, tratada, atip_partida, atip_chegada = cur.fetchone()

resumo = pd.DataFrame({
    "Indicador": ["Registros brutos", "Duplicatas excedentes", "Registros rejeitados", "Registros tratados", "Diferenças partida >24h", "Diferenças chegada >24h"],
    "Quantidade": [bruta, duplicatas, rejeicoes, tratada, atip_partida, atip_chegada]
})
display(resumo)
assert bruta == duplicatas + rejeicoes + tratada, "Reconciliação de registros falhou!"
assert (bruta, duplicatas, rejeicoes, tratada) == (983414, 470, 18, 982926), "Contagens diferentes do diagnóstico de referência."
print("ELT 2019 VALIDADO: registros conciliados!")

,Indicador,Quantidade
0,Registros brutos,983414
1,Duplicatas excedentes,470
2,Registros rejeitados,18
3,Registros tratados,982926
4,Diferenças partida >24h,684
5,Diferenças chegada >24h,690


ELT 2019 VALIDADO: registros conciliados!


## 11. Consultas para evidências do relatório
Exibe amostras da tabela tratada, ocorrências rejeitadas e distribuição por situação do voo. Esses resultados podem gerar prints e tabelas para o relatório.

In [11]:
consultas = {
    "Amostra tratada": "SELECT companhia_icao, numero_voo, partida_prevista, partida_real, atraso_partida_min, aeroporto_origem, aeroporto_destino, arquivo_origem, linha_origem FROM elt_2019.vra_tratada ORDER BY arquivo_origem, linha_origem LIMIT 8",
    "Rejeições": "SELECT arquivo_origem, linha_origem, motivo_rejeicao FROM elt_2019.vra_rejeicoes ORDER BY arquivo_origem, linha_origem LIMIT 18",
    "Situação dos voos": "SELECT situacao_voo, COUNT(*) AS registros FROM elt_2019.vra_tratada GROUP BY 1 ORDER BY registros DESC",
    "Total por mês (origem do arquivo)": "SELECT arquivo_origem, COUNT(*) AS registros FROM elt_2019.vra_tratada GROUP BY 1 ORDER BY 1"
}
for titulo, consulta in consultas.items():
    print("\n###", titulo)
    display(pd.read_sql_query(consulta, conn))


### Amostra tratada


/tmp/ipykernel_416/1083352613.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql_query(consulta, conn))


,companhia_icao,numero_voo,partida_prevista,partida_real,atraso_partida_min,aeroporto_origem,aeroporto_destino,arquivo_origem,linha_origem
0,AAF,0035,2019-01-25 06:15:00,2019-01-25 06:15:00,0.0,LFPO,SBKP,Base de dados 2019/VRA_2019_01.csv,2
1,AAF,0036,2019-01-25 20:15:00,2019-01-25 20:15:00,0.0,SBKP,LFPO,Base de dados 2019/VRA_2019_01.csv,3
2,AAF,0037,2019-01-26 09:45:00,2019-01-26 09:45:00,0.0,LFPO,SBKP,Base de dados 2019/VRA_2019_01.csv,4
3,AAF,0038,2019-01-26 23:45:00,2019-01-26 23:45:00,0.0,SBKP,LFPO,Base de dados 2019/VRA_2019_01.csv,5
4,AAF,0035,2019-01-27 06:15:00,2019-01-27 06:15:00,0.0,LFPO,SBKP,Base de dados 2019/VRA_2019_01.csv,6
5,AAF,0036,2019-01-27 20:15:00,2019-01-27 20:15:00,0.0,SBKP,LFPO,Base de dados 2019/VRA_2019_01.csv,7
6,AAF,0035,2019-01-29 06:15:00,2019-01-29 06:15:00,0.0,LFPO,SBKP,Base de dados 2019/VRA_2019_01.csv,8
7,AAF,0036,2019-01-29 20:15:00,2019-01-29 20:15:00,0.0,SBKP,LFPO,Base de dados 2019/VRA_2019_01.csv,9



### Rejeições


/tmp/ipykernel_416/1083352613.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql_query(consulta, conn))


,arquivo_origem,linha_origem,motivo_rejeicao
0,Base de dados 2019/VRA_2019_04.csv,76781,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
1,Base de dados 2019/VRA_2019_04.csv,76795,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
2,Base de dados 2019/VRA_2019_06.csv,26242,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
3,Base de dados 2019/VRA_2019_06.csv,51904,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
4,Base de dados 2019/VRA_2019_07.csv,30704,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
5,Base de dados 2019/VRA_2019_07.csv,61667,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
6,Base de dados 2019/VRA_2019_07.csv,62414,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
7,Base de dados 2019/VRA_2019_07.csv,62427,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
8,Base de dados 2019/VRA_2019_07.csv,63156,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES
9,Base de dados 2019/VRA_2019_07.csv,63170,PARTIDA_E_CHEGADA_PREVISTAS_AUSENTES


... (2 outputs left out)

,situacao_voo,registros
0,REALIZADO,964944
1,CANCELADO,17974
2,NÃO INFORMADO,8



### Total por mês (origem do arquivo)


/tmp/ipykernel_416/1083352613.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql_query(consulta, conn))


,arquivo_origem,registros
0,Base de dados 2019/VRA_2019_01.csv,91842
1,Base de dados 2019/VRA_2019_02.csv,79526
2,Base de dados 2019/VRA_2019_03.csv,83116
3,Base de dados 2019/VRA_2019_04.csv,77430
4,Base de dados 2019/VRA_2019_05.csv,76701
5,Base de dados 2019/VRA_2019_06.csv,74189
6,Base de dados 2019/VRA_2019_07.csv,85669
7,Base de dados 2019/VRA_2019_08.csv,80556
8,Base de dados 2019/VRA_2019_09.csv,80456
9,Base de dados 2019/VRA_2019_10.csv,84239


## 12. Conclusão da etapa e próximos passos

**Entregáveis desta fase:** notebook ELT 2019, SQL de transformação e resultados de validação (células de auditoria). Os dados originais ficaram preservados em `elt_2019.vra_bruta`. O Data Warehouse dimensional será carregado **posteriormente**, junto com as outras duplas, após consolidar as dimensões dos três anos.

**Para colocar no GitHub:** notebook + script SQL + contrato de dados + README explicando como executar no Colab. Não versione senhas nem o banco temporário. Os CSVs brutos podem ser obtidos na fonte ANAC.

**Nota:** a implementação separa os dados inadequados por regra contratual; preserva os registros rejeitados e as duplicatas excedentes em tabelas de auditoria, embora não entrem na tabela analítica.
